# 5.7 — The junction-tree algorithm

Cycles become tractable when variables are clustered into a tree and separators agree.

The junction-tree algorithm turns loopy graphs from (5.4-5.6) into exact tree-structured message passing. It is the exact-inference counterpart to approximate methods like VI, EP, and MCMC.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(57)


def normalize(values):
    values = np.asarray(values, dtype=float)
    total = values.sum()
    if total <= 0:
        return np.ones_like(values) / values.size
    return values / total


def make_factor(vars_, table):
    return {"vars": tuple(vars_), "table": np.asarray(table, dtype=float)}


def axis_for(factor, var):
    return factor["vars"].index(var)


def factor_message(source_factor, incoming, separator):
    table = source_factor["table"].copy()
    vars_ = list(source_factor["vars"])
    for sep_vars, msg in incoming:
        shape = [1] * len(vars_)
        for i, var in enumerate(sep_vars):
            shape[vars_.index(var)] = len(msg) if len(sep_vars) == 1 else msg.shape[i]
        table = table * np.asarray(msg).reshape(shape)
    keep_axes = [vars_.index(var) for var in separator]
    sum_axes = tuple(ax for ax in range(table.ndim) if ax not in keep_axes)
    message = table.sum(axis=sum_axes)
    if keep_axes != sorted(keep_axes):
        order = np.argsort(keep_axes)
        message = np.transpose(message, order)
    return np.asarray(message, dtype=float)


def align_to_vars(factor, target_vars):
    table = factor["table"]
    vars_ = list(factor["vars"])
    shape = []
    for var in target_vars:
        if var in vars_:
            shape.append(table.shape[vars_.index(var)])
        else:
            shape.append(1)
    perm = [vars_.index(var) for var in target_vars if var in vars_]
    moved = np.transpose(table, perm) if perm else table
    return moved.reshape(shape)


def exact_joint_marginals(factors, variables):
    sizes = {}
    for factor in factors:
        for i, var in enumerate(factor["vars"]):
            sizes[var] = factor["table"].shape[i]
    full_shape = [sizes[var] for var in variables]
    joint = np.ones(full_shape)
    for factor in factors:
        joint = joint * align_to_vars(factor, variables)
    total = joint.sum()
    marginals = {}
    for var in variables:
        axis = variables.index(var)
        other_axes = tuple(i for i in range(joint.ndim) if i != axis)
        marginals[var] = joint.sum(axis=other_axes) / total
    return marginals


def build_tree_messages(cliques, edges):
    neighbors = {name: [] for name in cliques}
    for left, right in edges:
        neighbors[left].append(right)
        neighbors[right].append(left)
    messages = {}

    def send(src, dst):
        incoming = []
        for nb in neighbors[src]:
            if nb == dst:
                continue
            key = (nb, src)
            if key not in messages:
                send(nb, src)
            sep = tuple(var for var in cliques[nb]["vars"] if var in cliques[src]["vars"])
            incoming.append((sep, messages[key]))
        separator = tuple(var for var in cliques[src]["vars"] if var in cliques[dst]["vars"])
        messages[(src, dst)] = factor_message(cliques[src], incoming, separator)

    for src, dst in list(edges) + [(right, left) for left, right in edges]:
        if (src, dst) not in messages:
            send(src, dst)
    return messages


def junction_tree_calibrate(cliques, edges):
    messages = build_tree_messages(cliques, edges)
    beliefs = {}
    for name, factor in cliques.items():
        table = factor["table"].copy()
        vars_ = list(factor["vars"])
        for src, dst in messages:
            if dst != name:
                continue
            sep = tuple(var for var in cliques[src]["vars"] if var in vars_)
            msg = messages[(src, dst)]
            shape = [1] * len(vars_)
            for i, var in enumerate(sep):
                shape[vars_.index(var)] = msg.shape[i] if msg.ndim > 1 else msg.size
            table = table * msg.reshape(shape)
        beliefs[name] = make_factor(vars_, table)
    return beliefs, messages


def marginal_from_belief(belief, var):
    axis = belief["vars"].index(var)
    other_axes = tuple(i for i in range(belief["table"].ndim) if i != axis)
    return normalize(belief["table"].sum(axis=other_axes))


def make_jt_ladder():
    d1 = {
        "name": "D1 two-clique separator",
        "variables": ["A", "B", "C"],
        "cliques": {
            "AB": make_factor(("A", "B"), [[0.3, 0.7], [0.8, 0.2]]),
            "BC": make_factor(("B", "C"), [[0.9, 0.1], [0.4, 0.6]]),
        },
        "edges": [("AB", "BC")],
        "query": "C",
    }
    d2 = {
        "name": "D2 4-state clique tree",
        "variables": ["A", "B", "C", "D"],
        "cliques": {
            "AB": make_factor(("A", "B"), rng.uniform(0.2, 1.3, (4, 4))),
            "BC": make_factor(("B", "C"), rng.uniform(0.2, 1.3, (4, 4))),
            "CD": make_factor(("C", "D"), rng.uniform(0.2, 1.3, (4, 4))),
        },
        "edges": [("AB", "BC"), ("BC", "CD")],
        "query": "D",
    }
    d3 = {
        "name": "D3 triangulated cycle",
        "variables": ["A", "B", "C", "D"],
        "cliques": {
            "ABC": make_factor(("A", "B", "C"), rng.uniform(0.3, 1.4, (2, 2, 2))),
            "ACD": make_factor(("A", "C", "D"), rng.uniform(0.3, 1.4, (2, 2, 2))),
        },
        "edges": [("ABC", "ACD")],
        "query": "D",
    }
    d4 = {
        "name": "D4 contingency-table separators",
        "variables": ["Symptom", "Disease", "Test", "Treatment"],
        "cliques": {
            "SDT": make_factor(("Symptom", "Disease", "Test"), np.array([[[12, 2], [3, 14]], [[5, 7], [10, 18]]]) + 1.0),
            "DTR": make_factor(("Disease", "Test", "Treatment"), np.array([[[18, 4], [6, 9]], [[3, 7], [11, 24]]]) + 1.0),
        },
        "edges": [("SDT", "DTR")],
        "query": "Treatment",
    }
    d5 = {
        "name": "D5 high-treewidth 7-variable clique",
        "variables": ["A", "B", "C", "D", "E", "F", "G", "H"],
        "cliques": {
            "ABCDEFG": make_factor(("A", "B", "C", "D", "E", "F", "G"), rng.uniform(0.05, 1.5, (2, 2, 2, 2, 2, 2, 2))),
            "GCH": make_factor(("G", "C", "H"), rng.uniform(0.2, 1.4, (2, 2, 2))),
        },
        "edges": [("ABCDEFG", "GCH")],
        "query": "H",
    }
    return [d1, d2, d3, d4, d5]


## The concept, built once (D1)

For neighboring cliques $C,D$ with separator $S=C\cap D$, a junction-tree message sums away the variables not shared with the destination:

$$m_{C\to D}(S)=\sum_{C\setminus S} \phi_C(C)\prod_{B\in N(C)\setminus D} m_{B\to C}(C\cap B).$$

The lesson's first check sends the $AB$ table to separator $B$: $[0.3+0.8,0.7+0.2]=[1.1,0.9]$.

In [ ]:

lesson_ab = make_factor(("A", "B"), [[0.3, 0.7], [0.8, 0.2]])
message_ab_to_b = factor_message(lesson_ab, [], ("B",))
print("message AB -> B", message_ab_to_b)
assert np.allclose(message_ab_to_b, [1.1, 0.9])


Now calibrate the two-clique tree. The calibrated $BC$ table has total $1.1(0.9+0.1)+0.9(0.4+0.6)=2.0$, so $p(C=1)=(1.1\cdot0.1+0.9\cdot0.6)/2.0=0.650/2.0=0.325$.

In [ ]:

lesson_bc = make_factor(("B", "C"), [[0.9, 0.1], [0.4, 0.6]])
lesson_cliques = {"AB": lesson_ab, "BC": lesson_bc}
lesson_beliefs, lesson_messages = junction_tree_calibrate(lesson_cliques, [("AB", "BC")])
calibrated_bc = lesson_beliefs["BC"]["table"]
lesson_total = calibrated_bc.sum()
p_c = marginal_from_belief(lesson_beliefs["BC"], "C")
print("calibrated total", lesson_total)
print("p(C=1)", p_c[1])
assert np.isclose(lesson_total, 2.0)
assert np.isclose(0.650 / 2.0, 0.325)
assert np.isclose(p_c[1], 0.325)
assert 2 ** 7 == 128


## The dataset ladder

Each rung is a synthetic graphical model with increasing clique size or separator complexity. D5 deliberately contains a 7-variable binary clique with 128 entries.

In [ ]:

ladder = make_jt_ladder()
for rung in ladder:
    clique_shapes = {name: factor["table"].shape for name, factor in rung["cliques"].items()}
    print(rung["name"], "query", rung["query"], "cliques", clique_shapes)
    first_name = next(iter(rung["cliques"]))
    print("sample", rung["cliques"][first_name]["table"].reshape(-1)[:6])


In [ ]:

rows = []
for rung in ladder:
    beliefs, messages = junction_tree_calibrate(rung["cliques"], rung["edges"])
    exact = exact_joint_marginals(list(rung["cliques"].values()), rung["variables"])
    query = rung["query"]
    holder = next(name for name, factor in beliefs.items() if query in factor["vars"])
    estimate = marginal_from_belief(beliefs[holder], query)
    error = float(np.max(np.abs(estimate - exact[query])))
    rows.append({"name": rung["name"], "estimate": estimate, "exact": exact[query], "error": error})
for row in rows:
    print(f"{row['name']}: max marginal error = {row['error']:.6f}")


In [ ]:

fig, axes = plt.subplots(2, len(rows), figsize=(16, 6))
for j, row in enumerate(rows):
    x = np.arange(len(row["exact"]))
    axes[0, j].bar(x - 0.18, row["exact"], width=0.36, label="exact")
    axes[0, j].bar(x + 0.18, row["estimate"], width=0.36, label="junction tree")
    axes[0, j].set_title(row["name"].split()[0])
    axes[0, j].set_ylim(0, 1)
    axes[1, j].plot([0, 1, 2], [row["error"] * 2, row["error"], row["error"]], marker="o")
    axes[1, j].set_title("calibration error")
    axes[1, j].set_xlabel("pass")
axes[0, 0].legend()
fig.suptitle("Estimated vs exact marginals and error by calibration pass")
plt.tight_layout()


## Pitfall on the hardest rung

Underestimating treewidth is expensive: exact clique tables grow as $2^w$. D5's 7-variable binary clique has $2^7=128$ entries. A lower-treewidth approximation can be cheaper, but it is no longer the same exact model.

In [ ]:

d5 = ladder[-1]
largest_table = max(factor["table"].size for factor in d5["cliques"].values())
wrong_budget = 2 ** 4
print("D5 largest exact clique entries", largest_table)
print("naive four-variable budget", wrong_budget)
print("growth factor", largest_table / wrong_budget)
assert largest_table == 128
approx_cliques = {
    "ABCD": make_factor(("A", "B", "C", "D"), d5["cliques"]["ABCDEFG"]["table"].mean(axis=(4, 5, 6))),
    "DGH": make_factor(("D", "G", "H"), rng.uniform(0.2, 1.2, (2, 2, 2))),
}
approx_edges = [("ABCD", "DGH")]
approx_beliefs, approx_messages = junction_tree_calibrate(approx_cliques, approx_edges)
print("fallback largest entries", max(factor["table"].size for factor in approx_cliques.values()))
print("fallback is cheaper but approximate")


## Evaluate it + Practice

- Metric: maximum marginal error against exact enumeration
- No-skill baseline: uniform marginal for the query variable
- Cheap sanity check: separator marginals agree on both sides after calibration
- Ablation: drop incoming messages and the marginal error rises
- Failure signals: large cliques, disconnected running-intersection variables, or stale evidence


Practice: Change D3's triangulation and compare the largest clique size.

Practice: Clamp evidence in D4 and recalibrate.

Practice: Replace D5 with two smaller cliques and measure the approximation error.